# Retail Performance Analytics

Synthetic 2025 data. This notebook runs the reproducible pipeline, then inspects quality, revenue, categories and customer segments. All numbers are demonstration results, not actual client outcomes.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'src/analyze.py').exists():
    ROOT = ROOT / 'portfolio/retail-performance'
assert (ROOT / 'src/analyze.py').exists(), 'Start in the project or repository folder'
sys.path.insert(0, str(ROOT / 'src'))
from analyze import main
main()

{
  "quality": {
    "raw_rows": 1220,
    "exact_duplicates_removed": 20,
    "missing_regions_labeled": 12,
    "quarantined_rows": 6,
    "valid_orders": 1194
  },
  "kpis": {
    "net_revenue_inr": 4677664.75,
    "gross_profit_inr": 1433298.67,
    "gross_margin_pct": 30.641329522386144,
    "completed_orders": 1034,
    "return_rate_pct": 8.251996450754215,
    "average_order_value_inr": 4523.853723404255,
    "purchasing_customers": 329,
    "repeat_customer_share_pct": 84.80243161094225
  },
  "reconciliation": "passed"
}


## Monthly performance

The first month has no preceding-month comparison. Completed-order revenue excludes final returned and cancelled statuses.

In [2]:
monthly = pd.read_csv(ROOT / 'reports/monthly_performance.csv')
print(monthly.to_string(index=False))

  month  placed_orders  completed_orders  returned_orders  net_revenue_inr  gross_profit_inr  previous_revenue  revenue_mom_pct
2025-01             91                81                4        359740.75         111159.88               NaN              NaN
2025-02            102                90                8        442277.40         143966.85         359740.75        22.943370
2025-03             84                79                2        377428.20         110076.66         442277.40       -14.662562
2025-04             92                77               10        296999.65          93973.54         377428.20       -21.309629
2025-05             93                77               11        379161.70         112188.70         296999.65        27.664022
2025-06            116                99                8        428191.00         126869.02         379161.70        12.930974
2025-07            104                92                7        441184.65         134092.56         428

## Category economics

Compare absolute profit and margin together. Return rates exclude cancellations; no causal category effect is implied.

In [3]:
categories = pd.read_csv(ROOT / 'reports/category_performance.csv')
print(categories.sort_values('gross_profit_inr', ascending=False).to_string(index=False))

   category  placed_orders  completed_orders  net_revenue_inr  gross_profit_inr  gross_margin_pct  return_rate_pct
Electronics            300               256       2186414.55         475147.35         21.731805         9.219858
    Fitness            328               285       1246889.90         433229.30         34.744792         8.945687
       Home            299               261        873224.65         323738.95         37.073959         5.434783
Accessories            267               232        371135.65         201183.07         54.207423         9.375000


## Customer activity

Recency is measured at 1 Jan 2026. Repeat behavior is observed within calendar 2025 only.

In [4]:
customers = pd.read_csv(ROOT / 'reports/customer_segments.csv')
print(customers.groupby('segment').agg(customers=('customer_id','count'), revenue_inr=('monetary_inr','sum')).to_string())

                   customers  revenue_inr
segment                                  
Active occasional         51    362904.40
Active repeat            135   2611942.10
Lapsed                   143   1702818.25


## Dashboard

![Four charts](reports/dashboard.svg)

## Conclusions and limits

Read [the generated findings](reports/findings.md). Profit excludes shipping, acquisition, overhead and return handling. Synthetic data cannot support market conclusions. A real retention intervention would need a randomized holdout to estimate incremental outcomes.